<a href="https://colab.research.google.com/github/yashikasgh/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashikasgh/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents the **daily performance of one content page for one client**. The main table used for this project is `fact_content_daily_performance`, where each row corresponds to a specific **content page, client, and report date**.

For this notebook, I will use **March 2026** as the time window because it is a **mid-panel month**. This follows the project guideline of avoiding the final month (June 2026) while exploring features and developing the analysis.


# Verifying One Content Page for One client.

In [22]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,cnt


#2. March 2026 time window + row count





In [23]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03';
""").df()

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


#3. availability (IS TRUE)

In [24]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
AND ga4_data_available IS TRUE;
""").df()

,available_rows
0,413966


In [25]:
!pip install -q datasets duckdb huggingface_hub pyarrow

In [26]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(HF_TOKEN[:8], "...")

hf_ekaqU ...


In [27]:
from huggingface_hub import login

login(HF_TOKEN)

In [28]:
import duckdb

In [29]:
con = duckdb.connect()

In [30]:
con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [31]:
rel = "hf://datasets/FlyRank/internship-warehouse"

In [32]:
con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 5;
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [33]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet');
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


# 2. Fields: feature / label / context / excluded

##1. Features (used by the model)
These are signals that are available before making a refresh decision.

*    **gsc_impressions**- Shows how much search visibility the page receives.
*   **gsc_clicks**- Shows the number of clicks generated from search.
*   **gsc_avg_position** - Shows the page's average search ranking position.
*   **ga4_pageviews**- Shows overall page traffic performance.
*   **ga4_engaged_sessions**- Shows how users interact with the page.

##2.Label / Proxy
Future improvement after refresh - The ideal label would be an observed increase in clicks, impressions, or engagement after a refresh. This is not available in the current dataset.

##3. Context
These fields help identify or group the data but should not be used as model features.

1.  **client_hash_id-**  Used to identify clients and group/split data, not for learning patterns.
2. **content_hash_id**-  Used to identify content items, not as a predictive feature.
3.  **report_date** - Used for filtering time windows and analysis.

##4. Excluded

1.  ** trend_direction**-  Derived from performance trends and can leak future information into the model.
2. **trend_pct **- Used to calculate trend information, so it can create leakage.
3. **Any future performance information** - Not available at the decision moment and would make evaluation unrealistic.


## 3. Verify it with queries (grain, counts, missing values, windows)

##1. Claim: One row represents one content page for one client on one day.





In [34]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


##2. Row count + date window verification

In [35]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03';
""").df()

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


##3. Availability verification

In [36]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
AND ga4_data_available IS TRUE;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,413966


##4. Missing values check

In [37]:
con.sql(f"""
SELECT
    AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS missing_impressions,
    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS missing_clicks,
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS missing_position,
    AVG(CASE WHEN ga4_pageviews IS NULL THEN 1.0 ELSE 0 END) AS missing_pageviews,
    AVG(CASE WHEN ga4_engaged_sessions IS NULL THEN 1.0 ELSE 0 END) AS missing_engagement
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03';
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,missing_impressions,missing_clicks,missing_position,missing_pageviews,missing_engagement
0,0.0,0.0,0.633074,0.30674,0.30674


## 4. Data limits

This dataset has some limitations that should be considered while building the analysis.

- The history available for each client is not the same. Some clients have more historical data than others, so comparisons across all clients may not always be equal.

- Some early rows may contain only GSC data because GA4 data was not available for those periods. Missing GA4 values should not always be treated as zero because they may represent unavailable data.

- Different tables may use different time windows. When combining data from multiple sources, the time periods need to be aligned carefully to avoid using information from the wrong time window.

- This data can help identify patterns and rank content opportunities, but it cannot prove that refreshing a page will directly cause better performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.